In [2]:
import torch
import torch.nn as nn
import numpy as np

Softmax:
- $S(y_i) = \frac {e^{y_i}}{ \sum {e^{y_j}}}$
               
                  Scores/Logits                 Probabilities
      |Linear| -> 2.0, 1.0, 0.1 -> |Softmax| -> 0.7, 0.2, 0.1 -> y_pred

- Applies exponential fnct to each element and normalizes it by dividing it with the sum of these exponentials

- What it does: Squashes the output to be btwn 0-1 (Probabilites) 

In [7]:
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=0)

x = np.array([2.0, 1.0, 0.1])
outputs = softmax(x)
print("Softmax numpy: ", outputs)

x = torch.tensor([2.0, 1.0, 0.1])
outputs = torch.softmax(x, dim=0)
print("Softmax tensor:", outputs)

Softmax numpy:  [0.65900114 0.24243297 0.09856589]
Softmax tensor: tensor([0.6590, 0.2424, 0.0986])


Cross Entropy Loss Function:

$D(\hat{Y}, Y) = -\frac {1}{N}  \sum Y_i  \log{\hat{Y_i}} $

- Y is One-Hot Encoded Class labels
- $\hat{Y}$ is Probabilities (Softmax)

Example:

                   Good prediction       Low Entropy Loss
$Y = [1,0,0] , \hat {Y} = [0.7,0.2,0.1]$ -> $D(\hat{Y}, Y) = 0.35$

                    Bad prediciton       High Entropy Loss
$Y = [1,0,0] , \hat {Y} = [0.1,0.3,0.6]$ -> $D(\hat{Y}, Y) = 2.30$

- Alot of times combined with Softmax
- Measures the performance of our classfiication model whose output is a prob. of 0-1 and used in multiclass problems,
- Loss increases as the predicted probability diverges from actual label. the better our pred, lower our loss


In [ ]:
def cross_entropy(actual, predicted):
    loss = -np.sum(actual * np.log(predicted))
    return loss # / float(predicted.shape[0]): we could normalize but we dont do it here

# y must be one hot encoded
# If class 0: [1 0 0]
# If class 1: [0 1 0]
# If class 2: [0 0 1]
Y = np.array([1,0,0])

# y_pred has probabolities
y_pred_good = np.array([0.7,0.2,0.1])
y_pred_bad = np.array([0.1,0.3,0.6])
l1 = cross_entropy(Y,y_pred_good)
l2 = cross_entropy(Y,y_pred_bad)
print(f'Loss1 numpy: {l1:.4f}')
print(f'Loss2 numpy: {l2:.4f}')

'''
# actual = [1, 0, 0]  (class 0, one-hot)
# predicted = [0.7, 0.2, 0.1]

# elementwise multiply:
# [1xlog(0.7), 0xlog(0.2), 0xlog(0.1)]
# = [log(0.7), 0, 0]         ← zeros wipe out wrong class scores
# = [-0.3567, 0, 0]
# sum = -0.3567, negate = 0.3567  ← only the correct class contributes
'''


Loss1 numpy: 0.3567
Loss2 numpy: 2.3026


For nn.CrossEntropyLoss:
- It already applies nn.LogSoftmax + nnNLLLoss (negative log likelihood loss)
- Remember: No Softmax in last layer
- Y has class labels, not One-Hot!
- Y_pred has raw scores (logits), no Softmax!

In [ ]:
loss = nn.CrossEntropyLoss()

# 1 sample:
#Y = torch.tensor([0])
# 3 Samples:
Y = torch.tensor([2,0,1])
# nsamples n nclasses = 3x3
Y_pred_good = torch.tensor([[0.1, 1.0, 2.1], [2.0, 1.0, 0.1], [0.1, 3.0, 0.1]]) # These are raw values, not applied to softmax
Y_pred_bad = torch.tensor([[2.1, 1.0, 0.1], [0.1, 1.0, 2.1], [0.1, 3.0, 0.1]])

l1 = loss(Y_pred_good, Y)
l2 = loss(Y_pred_bad, Y)

print(f'Loss1: {l1.item():.4f}')
print(f'Loss2: {l2.item():.4f}')

_, prediction1 = torch.max(Y_pred_good, 1) # _ discards the values: tensor([2.1, 2.0, 3.0])
_, prediction2 = torch.max(Y_pred_bad, 1) # dim =1 means find max across columns 

print(f'Pred1: {prediction1}')
print(f'Pred2: {prediction2}')


Loss1: 0.3018
Loss2: 1.6242
Pred1: tensor([2, 0, 1])
Pred2: tensor([0, 2, 1])


In [24]:
'''
Neural Net with Softmax
Which Animal? -> Multiclass problem:
             Activation, etc
                   *
               *   *   * --(4)-> * -> 0.95 [Dog]
Dog picture -> *   *   *
               *   *   * --(1)-> * -> 0.05 [Cat]
                   *          Softmax
             Linear  Linear

In pytorch: Use nn.CrossEntropyLoss()
No Softmax at the end
'''
# MultiClass problem:
class NeuralNet2(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet2, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) # Linear layer
        self.relu = nn.ReLU() # Activation Layer
        self.linear2 = nn.Linear(hidden_size, num_classes) # Linear layer
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # no softmax at the end
        return out
    
model = NeuralNet2(input_size=28*28, hidden_size=5, num_classes=3)
criterion = nn.CrossEntropyLoss() # applies softmax

'''
If we have a Binary classification probelm with 2 possible outputs,
then we rephrase question and change layers:
Is it a Dog? -> Binary problem
             Activation, etc 
                   *
               *   *    
Dog picture -> *   *   * --(4)-> * -> 0.90 [Yes]
               *   *          Sigmoid
                   *          
             Linear  Linear

In Pytorch: Use nn.BCELoss()
Sigmoid at the end
'''
# Binary classfication:
class NeuralNet1(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(NeuralNet1, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) # Linear layer
        self.relu = nn.ReLU() # Activation Layer
        self.linear2 = nn.Linear(hidden_size, 1) # Linear layer
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # sigmoid at the end
        y_pred = torch.sigmoid(out)
        return y_pred
    
model = NeuralNet1(input_size=28*28, hidden_size=5)
criterion = nn.BCELoss()


The whole forward pass pipeline:
- '*' means Numbers
- Input -> Neural network -> Raw Scores* -> Softmax -> Probabilities* -> CrossEntropyLoss -> Loss*

- Logits [0.1, 1.0, 2.1] ---[Softmax]---> probabilites [0.06, 0.24, 0.70] ---[CrossEntropyLoss]---> Loss = 0.36
- CrossEntropyLoss vs true label (Class 2 = [0, 0 1])